# Week 5 — Model Comparison: Decision Tree & Random Forest vs. Baseline

Continues from `03_baseline_model.ipynb`. Same `split()` / `prepare_data()` pipeline, extended to:
- run Linear Regression (baseline), Decision Tree, and Random Forest through one reusable training function
- compare test R² (plus MAE, MAPE, MdAPE) across all three
- log per-model behavior for the writeup

In [2]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, median_absolute_error

## 1. Split (unchanged from 03_baseline_model.ipynb)

In [3]:
def split(df, month):
    df["month-year"] = pd.to_datetime(df["YearMonth"])
    df = df.sort_values("month-year")
    latest_month = df["month-year"].max()
    print("Test month:", latest_month)

    test_df = df[df["month-year"] == latest_month]

    train_start = latest_month - pd.DateOffset(months=month)
    train_df = df[
        (df["month-year"] < latest_month) &
        (df["month-year"] >= train_start)
    ]

    print("Training period:", train_df["month-year"].min(), "to", train_df["month-year"].max())
    print("Testing period:", test_df["month-year"].min(), "to", test_df["month-year"].max())
    return train_df, test_df

## 2. Prepare data (unchanged)

In [8]:
def prepare_data(train_df, test_df):
    target = "ClosePrice"
    X_train = train_df.drop(columns=[target, "month-year", "YearMonth"])
    y_train = train_df[target]

    X_test = test_df.drop(columns=[target, "month-year", "YearMonth"])
    y_test = test_df[target]

    numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

    return X_train, y_train, X_test, y_test, numeric_cols, categorical_cols

## 3. Metrics helper

R² alone doesn't tell you much when it's negative — adding MAE (dollar-scale error), MAPE, and MdAPE (robust to outliers/skew, which real estate prices have plenty of) gives a fuller picture.

In [4]:
def compute_metrics(y_test, y_pred):
    return {
        "r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "mape": mean_absolute_percentage_error(y_test, y_pred),
        "mdape": np.median(np.abs((y_test - y_pred) / y_test))
    }

## 4. Reusable train/eval function

Same preprocessing (`StandardScaler` on numeric, `OneHotEncoder` on categorical), but the model itself is now a parameter instead of hardcoded — so Linear Regression, Decision Tree, and Random Forest all run through the identical pipeline. That keeps the comparison apples-to-apples: any difference in the numbers is coming from the model, not from different preprocessing.

Note: `numeric_cols`/`categorical_cols` are now explicit parameters (in `03_baseline_model.ipynb` they were read from globals, which only worked because of cell execution order).

In [9]:
def train_and_evaluate(model, model_name, X_train, y_train, X_test, y_test, numeric_cols, categorical_cols):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ]
    )

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = compute_metrics(y_test, y_pred)
    metrics["model"] = model_name

    print(f"{model_name:20s} | R\u00b2: {metrics['r2']:.4f} | MAE: {metrics['mae']:,.0f} | "
          f"MAPE: {metrics['mape']:.2%} | MdAPE: {metrics['mdape']:.2%}")

    return metrics, pipeline

## 5. Run all three models

In [14]:
df = pd.read_csv("data/df_cleaned.csv")
train_df, test_df = split(df, month=12)
X_train, y_train, X_test, y_test, numeric_cols, categorical_cols = prepare_data(train_df, test_df)

Test month: 2026-05-01 00:00:00
Training period: 2025-05-01 00:00:00 to 2026-04-01 00:00:00
Testing period: 2026-05-01 00:00:00 to 2026-05-01 00:00:00


In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, min_samples_leaf=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    metrics, pipeline = train_and_evaluate(
        model, name, X_train, y_train, X_test, y_test, numeric_cols, categorical_cols
    )
    results.append(metrics)
    fitted_pipelines[name] = pipeline

Linear Regression    | R²: -1.7198 | MAE: 909,050 | MAPE: 7766.55% | MdAPE: 90.26%
Decision Tree        | R²: -0.0528 | MAE: 414,971 | MAPE: 6631.08% | MdAPE: 51.95%
Random Forest        | R²: -0.0426 | MAE: 383,011 | MAPE: 12196.12% | MdAPE: 23.40%


## 6. Comparison table

In [16]:
comparison_df = pd.DataFrame(results)[["model", "r2", "mae", "mape", "mdape"]]
comparison_df = comparison_df.sort_values("r2", ascending=False).reset_index(drop=True)
comparison_df

,model,r2,mae,mape,mdape
0,Random Forest,-0.042580,383011.321112,121.961172,0.233993
1,Decision Tree,-0.052782,414970.631246,66.310792,0.519479
2,Linear Regression,-1.719775,909050.387129,77.665486,0.902550


## 7. Feature importance (tree-based models only)

Useful sanity check on *what* the tree/forest are actually keying off of — worth checking against your leakage TODO (ListPrice/OriginalListPrice) from the baseline notebook.

In [13]:
def get_feature_names(pipeline):
    return pipeline.named_steps["preprocessing"].get_feature_names_out()

for name in ["Decision Tree", "Random Forest"]:
    pipeline = fitted_pipelines[name]
    feature_names = get_feature_names(pipeline)
    importances = pipeline.named_steps["model"].feature_importances_

    top_features = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(10)
    )
    print(f"\nTop 10 features — {name}")
    print(top_features.to_string(index=False))


Top 10 features — Decision Tree
                                             feature  importance
                                      num__ListPrice    0.377719
             cat__HighSchoolDistrict_South Bay Union    0.131871
cat__BuyerOfficeName_Premier Agency Real Estate Inc.    0.106450
                    cat__ListAgentFirstName_Marianne    0.095701
                              num__OriginalListPrice    0.089426
                     cat__BuyerAgentLastName_Pacheco    0.077568
                       cat__ListAgentFirstName_Marty    0.046828
                  cat__ListAgentLastName_Castellanos    0.042649
                               cat__PostalCode_92113    0.031265
                                    num__LotSizeArea    0.000232

Top 10 features — Random Forest
                                            feature  importance
                                     num__ListPrice    0.142252
                                      num__Latitude    0.065500
                   cat__Buy

## 8. Model behavior notes

**Linear Regression (baseline)**
- Strength: fast, interpretable coefficients, no hyperparameter tuning needed.
- Weakness: assumes linear relationships between features and price; with 34 categorical columns one-hot encoded, the design matrix is high-dimensional and sparse, which linear models handle poorly — likely a contributor to the negative R² you saw in `03_baseline_model.ipynb`.

**Decision Tree**
- Strength: captures non-linear splits and interactions natively, no scaling required (though it doesn't hurt to keep it in the shared pipeline), interpretable via feature importance / tree structure.
- Weakness: prone to overfitting on a small training window (your `month=1`/`month=3` experiments), high variance — a single tree can swing a lot between runs. `max_depth`/`min_samples_leaf` above are just starting points; worth tuning.

**Random Forest**
- Strength: averaging across trees reduces the variance problem of a single Decision Tree, generally more robust test performance, gives a more stable feature importance ranking.
- Weakness: slower to train/predict, less directly interpretable than a single tree, still won't fix upstream data issues (outliers, leakage, insufficient training window) — it can only work with what's in the training data.

**Carried over from your baseline TODOs — still unresolved here, worth doing before trusting these numbers:**
- No outlier filtering on `ClosePrice` yet (0.5th/99.5th percentile thresholds, frozen from train, applied to test)
- `ListPrice`/`OriginalListPrice` leakage not yet addressed
- Not filtered to single-family residential only
- Only tested against `month=6`; worth re-running with `month=1` and `month=3` here too to see if the ranking between models holds